# Hassan Project 5 — Credit Risk & Loan Default Prediction with Explainable Machine Learning

**Python + SQL + Machine Learning + Explainability + Power BI**

Run the code cell below from top to bottom. It downloads the UCI dataset, performs the full analysis, trains four models, creates explainability outputs and packages everything into `Hassan_Project_5_Colab_Outputs.zip`.

> **Responsible-use note:** This is educational portfolio analytics only. Sensitive demographic fields are used for descriptive audit views but excluded from predictive modelling.

In [ ]:
# PROJECT 5 — CREDIT RISK & EXPLAINABLE ML
# Run this entire cell in Google Colab.

from pathlib import Path
import subprocess, sys, warnings, sqlite3, joblib, shutil
warnings.filterwarnings('ignore')

# Install the UCI helper package.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'ucimlrepo'])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ucimlrepo import fetch_ucirepo

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    roc_curve, precision_recall_curve
)
from sklearn.inspection import permutation_importance
from sklearn.utils.class_weight import compute_sample_weight

# -----------------------------------------------------------------------------
# 1. PROJECT FOLDERS
# -----------------------------------------------------------------------------
PROJECT_DIR = Path('project_5_outputs')
DATA_DIR = PROJECT_DIR / 'data'
IMAGES_DIR = PROJECT_DIR / 'images'
MODELS_DIR = PROJECT_DIR / 'models'
REPORTS_DIR = PROJECT_DIR / 'reports'
POWERBI_DIR = PROJECT_DIR / 'powerbi'
SQL_DIR = PROJECT_DIR / 'sql'
SRC_DIR = PROJECT_DIR / 'src'
SQL_RESULTS_DIR = PROJECT_DIR / 'sql_results'
for folder in [DATA_DIR, IMAGES_DIR, MODELS_DIR, REPORTS_DIR, POWERBI_DIR, SQL_DIR, SRC_DIR, SQL_RESULTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42

# -----------------------------------------------------------------------------
# 2. DOWNLOAD UCI DATASET 350
# -----------------------------------------------------------------------------
credit = fetch_ucirepo(id=350)
raw_df = pd.concat([credit.data.features.copy(), credit.data.targets.copy()], axis=1)
print('Raw shape:', raw_df.shape)
print('Raw columns:', list(raw_df.columns))

canonical_features = [
    'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE',
    'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6',
    'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6',
    'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6'
]

def clean_name(name):
    return str(name).strip().upper().replace(' ', '_').replace('-', '_')

df = raw_df.copy()
df.columns = [clean_name(c) for c in df.columns]

# Original spreadsheet versions sometimes expose an ID column; it is not a predictor.
if 'ID' in df.columns:
    df = df.drop(columns=['ID'])

# UCI package versions can expose X1...X23.
if all(f'X{i}' in df.columns for i in range(1, 24)):
    df = df.rename(columns={f'X{i}': canonical_features[i-1] for i in range(1, 24)})

if 'PAY_1' in df.columns and 'PAY_0' not in df.columns:
    df = df.rename(columns={'PAY_1': 'PAY_0'})

target_col = None
for candidate in ['DEFAULT_PAYMENT_NEXT_MONTH', 'DEFAULT_PAYMENT_NEXT_MONTH_', 'Y', 'DEFAULT']:
    if candidate in df.columns:
        target_col = candidate
        break
if target_col is None:
    target_col = df.columns[-1]

predictor_cols = [c for c in df.columns if c != target_col]
if not all(c in df.columns for c in canonical_features) and len(predictor_cols) == 23:
    df = df.rename(columns={old: new for old, new in zip(predictor_cols, canonical_features)})

df = df.rename(columns={target_col: 'DEFAULT'})
required = canonical_features + ['DEFAULT']
missing = [c for c in required if c not in df.columns]
if missing:
    raise RuntimeError('Could not standardise UCI schema. Missing columns: ' + str(missing))
df = df[required].copy()

# -----------------------------------------------------------------------------
# 3. CLEANING + DESCRIPTIVE LABELS
# -----------------------------------------------------------------------------
for c in df.columns:
    df[c] = pd.to_numeric(df[c], errors='coerce')

print('Missing cells before cleaning:', int(df.isna().sum().sum()))
print('Exact duplicates before cleaning:', int(df.duplicated().sum()))

df = df.drop_duplicates().copy()
df = df[
    df['LIMIT_BAL'].notna() & (df['LIMIT_BAL'] > 0) &
    df['AGE'].notna() & (df['AGE'] >= 18) &
    df['DEFAULT'].isin([0, 1])
].copy().reset_index(drop=True)

df.insert(0, 'CLIENT_ID', np.arange(1, len(df) + 1))

df['SexLabel'] = df['SEX'].map({1: 'Male', 2: 'Female'}).fillna('Other')
df['EducationLabel'] = df['EDUCATION'].map({
    1: 'Graduate School', 2: 'University', 3: 'High School',
    4: 'Other', 0: 'Other', 5: 'Other', 6: 'Other'
}).fillna('Other')
df['MarriageLabel'] = df['MARRIAGE'].map({1: 'Married', 2: 'Single', 3: 'Other', 0: 'Other'}).fillna('Other')
df['AgeGroup'] = pd.cut(
    df['AGE'], [17, 24, 34, 44, 54, 64, np.inf],
    labels=['18-24', '25-34', '35-44', '45-54', '55-64', '65+']
)
df['LimitBand'] = pd.cut(
    df['LIMIT_BAL'], [0, 50000, 100000, 200000, 300000, 500000, np.inf],
    labels=['<=50K', '50K-100K', '100K-200K', '200K-300K', '300K-500K', '500K+'],
    include_lowest=True
)

print('Clean rows:', len(df))
print('Observed defaults:', int(df['DEFAULT'].sum()))
print('Overall default rate:', f"{df['DEFAULT'].mean():.2%}")

# -----------------------------------------------------------------------------
# 4. FEATURE ENGINEERING
# -----------------------------------------------------------------------------
repay_cols = ['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']
bill_cols = [f'BILL_AMT{i}' for i in range(1, 7)]
payment_cols = [f'PAY_AMT{i}' for i in range(1, 7)]

df['AvgBillAmount'] = df[bill_cols].mean(axis=1)
df['AvgPaymentAmount'] = df[payment_cols].mean(axis=1)
df['TotalBillAmount'] = df[bill_cols].sum(axis=1)
df['TotalPaymentAmount'] = df[payment_cols].sum(axis=1)
df['RecentDelinquencyCount'] = (df[repay_cols] > 0).sum(axis=1)
df['SevereDelinquencyCount'] = (df[repay_cols] >= 2).sum(axis=1)
df['MaxRepaymentDelay'] = df[repay_cols].max(axis=1)
df['AvgRepaymentStatus'] = df[repay_cols].mean(axis=1)
df['RepaymentTrend'] = df['PAY_0'] - df['PAY_6']
df['BillUtilization'] = df['AvgBillAmount'] / df['LIMIT_BAL']
df['LatestUtilization'] = df['BILL_AMT1'] / df['LIMIT_BAL']
df['PaymentToBillRatio'] = df['TotalPaymentAmount'] / (df['TotalBillAmount'].abs() + 1)
df['RecentPaymentToBillRatio'] = (
    df[['PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3']].sum(axis=1) /
    (df[['BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3']].sum(axis=1).abs() + 1)
)
df['BillTrend'] = df['BILL_AMT1'] - df['BILL_AMT6']
df['PaymentTrend'] = df['PAY_AMT1'] - df['PAY_AMT6']
for c in ['BillUtilization', 'LatestUtilization', 'PaymentToBillRatio', 'RecentPaymentToBillRatio']:
    df[c] = df[c].clip(-5, 5)

# -----------------------------------------------------------------------------
# 5. EXPLORATORY CHARTS
# -----------------------------------------------------------------------------
summary = pd.DataFrame({
    'Status': ['Non-Default', 'Default'],
    'Clients': [(df['DEFAULT'] == 0).sum(), (df['DEFAULT'] == 1).sum()]
})
plt.figure(figsize=(7, 5))
plt.bar(summary['Status'], summary['Clients'])
plt.title('Default Distribution')
plt.ylabel('Clients')
plt.tight_layout()
plt.savefig(IMAGES_DIR / 'default_distribution.png', dpi=160, bbox_inches='tight')
plt.show()

limit_risk = df.groupby('LimitBand', observed=False).agg(
    Clients=('CLIENT_ID', 'count'), Defaults=('DEFAULT', 'sum'),
    DefaultRate=('DEFAULT', 'mean'), TotalCreditLimit=('LIMIT_BAL', 'sum')
).reset_index()
print('\nDefault by credit limit band:')
print(limit_risk)
plt.figure(figsize=(9, 5))
plt.bar(limit_risk['LimitBand'].astype(str), limit_risk['DefaultRate'] * 100)
plt.title('Default Rate by Credit Limit Band')
plt.ylabel('Default Rate (%)')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(IMAGES_DIR / 'default_by_limit_band.png', dpi=160, bbox_inches='tight')
plt.show()

def risk_chart(column, filename, title):
    t = df.groupby(column, observed=False)['DEFAULT'].agg(['count', 'mean']).reset_index()
    t = t.rename(columns={'count': 'Clients', 'mean': 'DefaultRate'})
    print('\n', title)
    print(t)
    plt.figure(figsize=(9, 5))
    plt.bar(t[column].astype(str), t['DefaultRate'] * 100)
    plt.title(title)
    plt.ylabel('Default Rate (%)')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.savefig(IMAGES_DIR / filename, dpi=160, bbox_inches='tight')
    plt.show()

risk_chart('AgeGroup', 'default_by_age_group.png', 'Default Rate by Age Group')
risk_chart('EducationLabel', 'default_by_education.png', 'Default Rate by Education')
risk_chart('MarriageLabel', 'default_by_marriage.png', 'Default Rate by Marital Status')
risk_chart('RecentDelinquencyCount', 'default_by_recent_delinquency.png', 'Default Rate by Recent Delinquency Count')
risk_chart('PAY_0', 'repayment_status.png', 'Default Rate by Most Recent Repayment Status')

# -----------------------------------------------------------------------------
# 6. SAVE CLEANED DATA + SQLITE
# -----------------------------------------------------------------------------
df.to_csv(DATA_DIR / 'credit_risk_cleaned.csv', index=False)
conn = sqlite3.connect(PROJECT_DIR / 'credit_risk.sqlite')
df.to_sql('credit_risk', conn, if_exists='replace', index=False)

SQL_QUERIES = {
    '01_portfolio_overview': "SELECT COUNT(*) Clients,SUM([DEFAULT]) Defaults,AVG([DEFAULT]) DefaultRate,SUM(LIMIT_BAL) TotalCreditLimit,AVG(LIMIT_BAL) AverageCreditLimit FROM credit_risk;",
    '02_default_by_limit_band': "SELECT LimitBand,COUNT(*) Clients,SUM([DEFAULT]) Defaults,AVG([DEFAULT]) DefaultRate,SUM(LIMIT_BAL) TotalCreditLimit FROM credit_risk GROUP BY LimitBand ORDER BY DefaultRate DESC;",
    '03_default_by_recent_status': "SELECT PAY_0,COUNT(*) Clients,SUM([DEFAULT]) Defaults,AVG([DEFAULT]) DefaultRate FROM credit_risk GROUP BY PAY_0 ORDER BY DefaultRate DESC;",
    '04_default_by_age_group': "SELECT AgeGroup,COUNT(*) Clients,AVG([DEFAULT]) DefaultRate FROM credit_risk GROUP BY AgeGroup ORDER BY DefaultRate DESC;",
    '05_default_by_education': "SELECT EducationLabel,COUNT(*) Clients,AVG([DEFAULT]) DefaultRate FROM credit_risk GROUP BY EducationLabel ORDER BY DefaultRate DESC;",
    '06_default_by_marriage': "SELECT MarriageLabel,COUNT(*) Clients,AVG([DEFAULT]) DefaultRate FROM credit_risk GROUP BY MarriageLabel ORDER BY DefaultRate DESC;",
    '07_repeated_delinquency': "SELECT CLIENT_ID,LIMIT_BAL,RecentDelinquencyCount,SevereDelinquencyCount,MaxRepaymentDelay,[DEFAULT] FROM credit_risk WHERE RecentDelinquencyCount>=3 ORDER BY RecentDelinquencyCount DESC,LIMIT_BAL DESC;",
    '08_bill_payment_by_default': "SELECT [DEFAULT],COUNT(*) Clients,AVG(AvgBillAmount) AvgBillAmount,AVG(AvgPaymentAmount) AvgPaymentAmount,AVG(PaymentToBillRatio) AvgPaymentToBillRatio,AVG(BillUtilization) AvgBillUtilization FROM credit_risk GROUP BY [DEFAULT];",
    '09_high_limit_late_payment': "SELECT CLIENT_ID,LIMIT_BAL,RecentDelinquencyCount,MaxRepaymentDelay,PaymentToBillRatio,[DEFAULT] FROM credit_risk WHERE LIMIT_BAL>=300000 AND RecentDelinquencyCount>=2 ORDER BY LIMIT_BAL DESC;",
    '10_delinquency_bands': "SELECT CASE WHEN RecentDelinquencyCount=0 THEN '0 months' WHEN RecentDelinquencyCount<=2 THEN '1-2 months' WHEN RecentDelinquencyCount<=4 THEN '3-4 months' ELSE '5-6 months' END DelinquencyBand,COUNT(*) Clients,AVG([DEFAULT]) DefaultRate,SUM(LIMIT_BAL) TotalCreditLimit FROM credit_risk GROUP BY DelinquencyBand ORDER BY DefaultRate DESC;",
    '11_payment_ratio_bands': "SELECT CASE WHEN PaymentToBillRatio<0.05 THEN '<5%' WHEN PaymentToBillRatio<0.10 THEN '5%-10%' WHEN PaymentToBillRatio<0.25 THEN '10%-25%' ELSE '25%+' END PaymentRatioBand,COUNT(*) Clients,AVG([DEFAULT]) DefaultRate FROM credit_risk GROUP BY PaymentRatioBand ORDER BY DefaultRate DESC;",
    '12_limit_rank_window': "SELECT CLIENT_ID,LIMIT_BAL,[DEFAULT],RANK() OVER (ORDER BY LIMIT_BAL DESC) CreditLimitRank FROM credit_risk ORDER BY CreditLimitRank LIMIT 100;"
}
for name, query in SQL_QUERIES.items():
    pd.read_sql_query(query, conn).to_csv(SQL_RESULTS_DIR / f'{name}.csv', index=False)
print('\nPortfolio SQL summary:')
print(pd.read_sql_query(SQL_QUERIES['01_portfolio_overview'], conn))

# -----------------------------------------------------------------------------
# 7. RESPONSIBLE MODEL FEATURE SET
# -----------------------------------------------------------------------------
excluded_from_model = [
    'CLIENT_ID', 'DEFAULT', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE',
    'SexLabel', 'EducationLabel', 'MarriageLabel', 'AgeGroup', 'LimitBand'
]
model_features = [
    c for c in df.columns
    if c not in excluded_from_model and pd.api.types.is_numeric_dtype(df[c])
]
X = df[model_features].replace([np.inf, -np.inf], np.nan).copy()
X = X.fillna(X.median(numeric_only=True))
y = df['DEFAULT'].astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)
print('\nPredictive features:', len(model_features))
print('Train rows:', len(X_train), 'Test rows:', len(X_test))

# -----------------------------------------------------------------------------
# 8. TRAIN FOUR MODELS
# -----------------------------------------------------------------------------
models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=3000, class_weight='balanced', random_state=RANDOM_STATE))
    ]),
    'Decision Tree': DecisionTreeClassifier(
        max_depth=6, min_samples_leaf=50, class_weight='balanced', random_state=RANDOM_STATE
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=10, min_samples_leaf=15,
        class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=150, learning_rate=0.05, max_depth=3, random_state=RANDOM_STATE
    )
}
weights = compute_sample_weight(class_weight='balanced', y=y_train)
fitted_models = {}
test_probabilities = {}
evaluation_rows = []

for name, model in models.items():
    print('Training:', name)
    if name == 'Gradient Boosting':
        model.fit(X_train, y_train, sample_weight=weights)
    else:
        model.fit(X_train, y_train)
    prob = model.predict_proba(X_test)[:, 1]
    pred = (prob >= 0.50).astype(int)
    fitted_models[name] = model
    test_probabilities[name] = prob
    evaluation_rows.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred, zero_division=0),
        'Recall': recall_score(y_test, pred, zero_division=0),
        'F1': f1_score(y_test, pred, zero_division=0),
        'ROC_AUC': roc_auc_score(y_test, prob),
        'PR_AUC': average_precision_score(y_test, prob)
    })

model_comparison = pd.DataFrame(evaluation_rows).sort_values(
    ['PR_AUC', 'ROC_AUC'], ascending=False
).reset_index(drop=True)
model_comparison.to_csv(REPORTS_DIR / 'model_comparison.csv', index=False)
print('\nModel comparison:')
print(model_comparison.round(3).to_string(index=False))
BEST_MODEL_NAME = model_comparison.iloc[0]['Model']
best_model = fitted_models[BEST_MODEL_NAME]
best_test_prob = test_probabilities[BEST_MODEL_NAME]
print('Selected model by PR-AUC:', BEST_MODEL_NAME)

# -----------------------------------------------------------------------------
# 9. MODEL COMPARISON / ROC / PR CURVES
# -----------------------------------------------------------------------------
plot_df = model_comparison.set_index('Model')[['ROC_AUC', 'PR_AUC', 'Recall', 'Precision', 'F1']]
x = np.arange(len(plot_df.index)); width = 0.15
plt.figure(figsize=(12, 6))
for i, metric in enumerate(plot_df.columns):
    plt.bar(x + (i - 2) * width, plot_df[metric].values, width=width, label=metric)
plt.xticks(x, plot_df.index, rotation=20, ha='right')
plt.ylim(0, 1); plt.title('Credit Risk Model Comparison'); plt.ylabel('Score'); plt.legend(); plt.tight_layout()
plt.savefig(IMAGES_DIR / 'model_comparison.png', dpi=160, bbox_inches='tight'); plt.show()

plt.figure(figsize=(9, 6))
for name, prob in test_probabilities.items():
    fpr, tpr, _ = roc_curve(y_test, prob)
    plt.plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_test, prob):.3f})")
plt.plot([0, 1], [0, 1], linestyle='--'); plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate'); plt.title('ROC Curves'); plt.legend(); plt.tight_layout()
plt.savefig(IMAGES_DIR / 'roc_curve.png', dpi=160, bbox_inches='tight'); plt.show()

plt.figure(figsize=(9, 6))
for name, prob in test_probabilities.items():
    precision_vals, recall_vals, _ = precision_recall_curve(y_test, prob)
    plt.plot(recall_vals, precision_vals, label=f"{name} (AP={average_precision_score(y_test, prob):.3f})")
plt.xlabel('Recall'); plt.ylabel('Precision'); plt.title('Precision-Recall Curves'); plt.legend(); plt.tight_layout()
plt.savefig(IMAGES_DIR / 'precision_recall_curve.png', dpi=160, bbox_inches='tight'); plt.show()

# -----------------------------------------------------------------------------
# 10. THRESHOLD ANALYSIS
# -----------------------------------------------------------------------------
threshold_rows = []
for threshold in np.arange(0.10, 0.81, 0.05):
    pred = (best_test_prob >= threshold).astype(int)
    threshold_rows.append({
        'Threshold': round(float(threshold), 2),
        'Precision': precision_score(y_test, pred, zero_division=0),
        'Recall': recall_score(y_test, pred, zero_division=0),
        'F1': f1_score(y_test, pred, zero_division=0),
        'PredictedHighRiskCount': int(pred.sum())
    })
threshold_df = pd.DataFrame(threshold_rows)
threshold_df.to_csv(REPORTS_DIR / 'threshold_analysis.csv', index=False)
BEST_F1_THRESHOLD = float(threshold_df.loc[threshold_df['F1'].idxmax(), 'Threshold'])
print('\nBest-F1 threshold:', BEST_F1_THRESHOLD)

plt.figure(figsize=(9, 6))
plt.plot(threshold_df['Threshold'], threshold_df['Precision'], label='Precision')
plt.plot(threshold_df['Threshold'], threshold_df['Recall'], label='Recall')
plt.plot(threshold_df['Threshold'], threshold_df['F1'], label='F1')
plt.xlabel('Classification Threshold'); plt.ylabel('Score'); plt.title(f'Threshold Trade-off — {BEST_MODEL_NAME}'); plt.legend(); plt.tight_layout()
plt.savefig(IMAGES_DIR / 'threshold_tradeoff.png', dpi=160, bbox_inches='tight'); plt.show()

cm = confusion_matrix(y_test, (best_test_prob >= BEST_F1_THRESHOLD).astype(int))
plt.figure(figsize=(6, 5)); plt.imshow(cm); plt.colorbar()
plt.xticks([0, 1], ['Non-Default', 'Default']); plt.yticks([0, 1], ['Non-Default', 'Default'])
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title(f'Confusion Matrix — {BEST_MODEL_NAME}')
for i in range(2):
    for j in range(2):
        plt.text(j, i, str(cm[i, j]), ha='center', va='center')
plt.tight_layout(); plt.savefig(IMAGES_DIR / 'confusion_matrix.png', dpi=160, bbox_inches='tight'); plt.show()

# -----------------------------------------------------------------------------
# 11. EXPLAINABLE ML — PERMUTATION IMPORTANCE
# -----------------------------------------------------------------------------
perm = permutation_importance(
    best_model, X_test, y_test, scoring='average_precision',
    n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1
)
feature_importance = pd.DataFrame({
    'Feature': model_features,
    'ImportanceMean': perm.importances_mean,
    'ImportanceStd': perm.importances_std
}).sort_values('ImportanceMean', ascending=False).reset_index(drop=True)
feature_importance.to_csv(REPORTS_DIR / 'feature_importance.csv', index=False)
print('\nTop feature importance:')
print(feature_importance.head(15).to_string(index=False))

top_imp = feature_importance.head(15).sort_values('ImportanceMean')
plt.figure(figsize=(10, 7)); plt.barh(top_imp['Feature'], top_imp['ImportanceMean'])
plt.xlabel('Permutation Importance'); plt.title(f'Top Model Drivers — {BEST_MODEL_NAME}'); plt.tight_layout()
plt.savefig(IMAGES_DIR / 'feature_importance.png', dpi=160, bbox_inches='tight'); plt.show()

# -----------------------------------------------------------------------------
# 12. REFIT SELECTED MODEL + PORTFOLIO RISK BANDS
# -----------------------------------------------------------------------------
final_model = models[BEST_MODEL_NAME]
if BEST_MODEL_NAME == 'Gradient Boosting':
    final_model.fit(X, y, sample_weight=compute_sample_weight(class_weight='balanced', y=y))
else:
    final_model.fit(X, y)
full_probability = final_model.predict_proba(X)[:, 1]

risk_predictions = df.copy()
risk_predictions['PredictedDefaultProbability'] = full_probability
risk_predictions['RiskBand'] = pd.cut(
    full_probability, [-np.inf, 0.20, 0.50, np.inf],
    labels=['Low', 'Medium', 'High'], right=False
)
risk_predictions['BestF1ThresholdFlag'] = (full_probability >= BEST_F1_THRESHOLD).astype(int)

risk_summary = risk_predictions.groupby('RiskBand', observed=False).agg(
    Clients=('CLIENT_ID', 'count'),
    AvgPredictedProbability=('PredictedDefaultProbability', 'mean'),
    ActualDefaultRate=('DEFAULT', 'mean'),
    TotalCreditLimit=('LIMIT_BAL', 'sum'),
    AvgCreditLimit=('LIMIT_BAL', 'mean')
).reset_index()
print('\nRisk-band summary:')
print(risk_summary.to_string(index=False))
risk_predictions.to_csv(DATA_DIR / 'credit_risk_predictions.csv', index=False)
risk_summary.to_csv(DATA_DIR / 'risk_band_summary.csv', index=False)
joblib.dump(final_model, MODELS_DIR / 'best_credit_risk_model.pkl')

plt.figure(figsize=(8, 5)); plt.bar(risk_summary['RiskBand'].astype(str), risk_summary['ActualDefaultRate'] * 100)
plt.title('Observed Default Rate by Predicted Risk Band'); plt.ylabel('Observed Default Rate (%)'); plt.tight_layout()
plt.savefig(IMAGES_DIR / 'risk_band_distribution.png', dpi=160, bbox_inches='tight'); plt.show()

# -----------------------------------------------------------------------------
# 13. POST-MODEL SQL + POWER BI EXPORTS
# -----------------------------------------------------------------------------
risk_predictions[['CLIENT_ID','LIMIT_BAL','DEFAULT','PredictedDefaultProbability','RiskBand','BestF1ThresholdFlag']].to_sql(
    'credit_risk_predictions', conn, if_exists='replace', index=False
)
POST_MODEL_SQL = {
    '13_risk_band_exposure': "SELECT RiskBand,COUNT(*) Clients,AVG(PredictedDefaultProbability) AvgPredictedProbability,AVG([DEFAULT]) ActualDefaultRate,SUM(LIMIT_BAL) TotalCreditLimit FROM credit_risk_predictions GROUP BY RiskBand ORDER BY AvgPredictedProbability;",
    '14_high_risk_clients': "SELECT CLIENT_ID,LIMIT_BAL,PredictedDefaultProbability,[DEFAULT] FROM credit_risk_predictions WHERE RiskBand='High' ORDER BY PredictedDefaultProbability DESC LIMIT 100;",
    '15_portfolio_risk_summary': "SELECT COUNT(*) Clients,AVG([DEFAULT]) ActualDefaultRate,AVG(PredictedDefaultProbability) AvgPredictedRisk,SUM(CASE WHEN RiskBand='High' THEN 1 ELSE 0 END) HighRiskClients,SUM(CASE WHEN RiskBand='High' THEN LIMIT_BAL ELSE 0 END) HighRiskCreditLimit FROM credit_risk_predictions;"
}
for name, query in POST_MODEL_SQL.items():
    pd.read_sql_query(query, conn).to_csv(SQL_RESULTS_DIR / f'{name}.csv', index=False)

powerbi_cols = [
    'CLIENT_ID','LIMIT_BAL','LimitBand','AGE','AgeGroup','SexLabel','EducationLabel','MarriageLabel',
    'PAY_0','PAY_2','PAY_3','PAY_4','PAY_5','PAY_6','AvgBillAmount','AvgPaymentAmount',
    'RecentDelinquencyCount','SevereDelinquencyCount','MaxRepaymentDelay','AvgRepaymentStatus','RepaymentTrend',
    'BillUtilization','LatestUtilization','PaymentToBillRatio','RecentPaymentToBillRatio','DEFAULT',
    'PredictedDefaultProbability','RiskBand','BestF1ThresholdFlag'
]
risk_predictions[powerbi_cols].to_csv(POWERBI_DIR / 'credit_risk_powerbi.csv', index=False)
model_comparison.to_csv(POWERBI_DIR / 'model_comparison.csv', index=False)
threshold_df.to_csv(POWERBI_DIR / 'threshold_analysis.csv', index=False)
feature_importance.to_csv(POWERBI_DIR / 'feature_importance.csv', index=False)
risk_summary.to_csv(POWERBI_DIR / 'risk_band_summary.csv', index=False)

# -----------------------------------------------------------------------------
# 14. MYSQL-READY SQL FILES
# -----------------------------------------------------------------------------
(SQL_DIR / 'database_setup.sql').write_text(
"""CREATE DATABASE IF NOT EXISTS credit_risk_analytics;
USE credit_risk_analytics;

CREATE TABLE IF NOT EXISTS credit_risk (
 client_id INT PRIMARY KEY,
 limit_bal DOUBLE,
 sex INT,
 education INT,
 marriage INT,
 age INT,
 pay_0 INT, pay_2 INT, pay_3 INT, pay_4 INT, pay_5 INT, pay_6 INT,
 bill_amt1 DOUBLE, bill_amt2 DOUBLE, bill_amt3 DOUBLE, bill_amt4 DOUBLE, bill_amt5 DOUBLE, bill_amt6 DOUBLE,
 pay_amt1 DOUBLE, pay_amt2 DOUBLE, pay_amt3 DOUBLE, pay_amt4 DOUBLE, pay_amt5 DOUBLE, pay_amt6 DOUBLE,
 default_flag INT,
 avg_bill_amount DOUBLE,
 avg_payment_amount DOUBLE,
 recent_delinquency_count INT,
 severe_delinquency_count INT,
 max_repayment_delay INT,
 avg_repayment_status DOUBLE,
 repayment_trend DOUBLE,
 bill_utilization DOUBLE,
 latest_utilization DOUBLE,
 payment_to_bill_ratio DOUBLE,
 recent_payment_to_bill_ratio DOUBLE
);
""", encoding='utf-8')

(SQL_DIR / 'cleaning_queries.sql').write_text(
"""SELECT COUNT(*) AS total_rows FROM credit_risk;
SELECT client_id, COUNT(*) AS duplicate_count FROM credit_risk GROUP BY client_id HAVING COUNT(*) > 1;
SELECT * FROM credit_risk WHERE limit_bal <= 0 OR age < 18 OR default_flag NOT IN (0,1);
""", encoding='utf-8')

(SQL_DIR / 'analysis_queries.sql').write_text(
"""SELECT COUNT(*) clients,SUM(default_flag) defaults,AVG(default_flag) default_rate,SUM(limit_bal) total_credit_limit FROM credit_risk;
SELECT pay_0,COUNT(*) clients,AVG(default_flag) default_rate FROM credit_risk GROUP BY pay_0 ORDER BY default_rate DESC;
SELECT recent_delinquency_count,COUNT(*) clients,AVG(default_flag) default_rate,SUM(limit_bal) total_credit_limit FROM credit_risk GROUP BY recent_delinquency_count;
SELECT default_flag,AVG(avg_bill_amount) avg_bill_amount,AVG(avg_payment_amount) avg_payment_amount,AVG(payment_to_bill_ratio) avg_payment_to_bill_ratio FROM credit_risk GROUP BY default_flag;
SELECT client_id,limit_bal,default_flag,RANK() OVER (ORDER BY limit_bal DESC) credit_limit_rank FROM credit_risk ORDER BY credit_limit_rank LIMIT 100;
""", encoding='utf-8')

# -----------------------------------------------------------------------------
# 15. REUSABLE PYTHON FEATURE PIPELINE
# -----------------------------------------------------------------------------
pipeline_source = """import pandas as pd
import numpy as np
REPAY_COLS=['PAY_0','PAY_2','PAY_3','PAY_4','PAY_5','PAY_6']
BILL_COLS=[f'BILL_AMT{i}' for i in range(1,7)]
PAYMENT_COLS=[f'PAY_AMT{i}' for i in range(1,7)]

def engineer_credit_features(df):
    df=df.copy()
    df['AvgBillAmount']=df[BILL_COLS].mean(axis=1)
    df['AvgPaymentAmount']=df[PAYMENT_COLS].mean(axis=1)
    df['TotalBillAmount']=df[BILL_COLS].sum(axis=1)
    df['TotalPaymentAmount']=df[PAYMENT_COLS].sum(axis=1)
    df['RecentDelinquencyCount']=(df[REPAY_COLS]>0).sum(axis=1)
    df['SevereDelinquencyCount']=(df[REPAY_COLS]>=2).sum(axis=1)
    df['MaxRepaymentDelay']=df[REPAY_COLS].max(axis=1)
    df['AvgRepaymentStatus']=df[REPAY_COLS].mean(axis=1)
    df['RepaymentTrend']=df['PAY_0']-df['PAY_6']
    df['BillUtilization']=df['AvgBillAmount']/df['LIMIT_BAL']
    df['LatestUtilization']=df['BILL_AMT1']/df['LIMIT_BAL']
    df['PaymentToBillRatio']=df['TotalPaymentAmount']/(df['TotalBillAmount'].abs()+1)
    df['RecentPaymentToBillRatio']=df[['PAY_AMT1','PAY_AMT2','PAY_AMT3']].sum(axis=1)/(df[['BILL_AMT1','BILL_AMT2','BILL_AMT3']].sum(axis=1).abs()+1)
    df['BillTrend']=df['BILL_AMT1']-df['BILL_AMT6']
    df['PaymentTrend']=df['PAY_AMT1']-df['PAY_AMT6']
    return df
"""
(SRC_DIR / 'credit_risk_pipeline.py').write_text(pipeline_source, encoding='utf-8')

# -----------------------------------------------------------------------------
# 16. BUSINESS FINDINGS + README + ZIP
# -----------------------------------------------------------------------------
best_row = model_comparison.iloc[0]
high = risk_summary[risk_summary['RiskBand'].astype(str) == 'High']
high_clients = int(high.iloc[0]['Clients']) if len(high) else 0
high_default_rate = float(high.iloc[0]['ActualDefaultRate']) if len(high) else np.nan
high_credit_limit = float(high.iloc[0]['TotalCreditLimit']) if len(high) else 0.0
top_features = feature_importance.head(10)['Feature'].tolist()

findings = f"""CREDIT RISK & EXPLAINABLE ML — BUSINESS FINDINGS
=================================================
Clients analysed: {len(df):,}
Observed defaults: {int(df['DEFAULT'].sum()):,}
Overall default rate: {df['DEFAULT'].mean():.2%}

Selected model by PR-AUC: {BEST_MODEL_NAME}
Accuracy: {best_row['Accuracy']:.3f}
Precision: {best_row['Precision']:.3f}
Recall: {best_row['Recall']:.3f}
F1: {best_row['F1']:.3f}
ROC-AUC: {best_row['ROC_AUC']:.3f}
PR-AUC: {best_row['PR_AUC']:.3f}

Best-F1 test threshold: {BEST_F1_THRESHOLD:.2f}
High-risk clients: {high_clients:,}
Observed default rate in High-risk band: {high_default_rate:.2%}
High-risk credit-limit exposure proxy: {high_credit_limit:,.0f}

Top model features: {', '.join(top_features)}

Responsible-use notes:
1. Educational portfolio analytics only.
2. Sensitive demographic fields are excluded from predictive modelling.
3. Accuracy alone is insufficient for an imbalanced target.
4. PR-AUC, ROC-AUC, recall, precision and threshold trade-offs should be reviewed together.
5. Feature importance describes model behaviour and does not establish causality.
6. Real lending use would require calibration, fairness testing, governance, monitoring and human oversight.
"""
(REPORTS_DIR / 'business_findings.txt').write_text(findings, encoding='utf-8')
print('\n' + findings)

(PROJECT_DIR / 'requirements.txt').write_text(
    'pandas\nnumpy\nmatplotlib\nscikit-learn\njoblib\nucimlrepo\n', encoding='utf-8'
)
(PROJECT_DIR / 'README_GENERATED.md').write_text(
    f"""# Credit Risk & Loan Default Prediction with Explainable ML

Dataset: UCI Default of Credit Card Clients

Records: {len(df):,}
Overall default rate: {df['DEFAULT'].mean():.2%}
Selected model: {BEST_MODEL_NAME}
Test ROC-AUC: {best_row['ROC_AUC']:.3f}
Test PR-AUC: {best_row['PR_AUC']:.3f}

Sensitive demographic variables are excluded from predictive modelling and used only for descriptive audit analysis.

Educational portfolio analytics only — not an automated lending decision system.
""", encoding='utf-8'
)

zip_path = shutil.make_archive('Hassan_Project_5_Colab_Outputs', 'zip', PROJECT_DIR)
print('\nCreated ZIP:', zip_path)
print('Project 5 analysis is complete.')
print('Download Hassan_Project_5_Colab_Outputs.zip from the Colab Files panel.')

## Finished

When the code finishes, open the Colab **Files** panel and download `Hassan_Project_5_Colab_Outputs.zip`. Upload that ZIP back to ChatGPT so the final report, Power BI project and GitHub package can be prepared.